# 04 · OLS Regression on Log Returns

Predict next-day log return from technical indicators using Ordinary Least Squares regression and evaluate residual diagnostics.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('__file__')), '..'))
import config

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import seaborn as sns

sns.set_theme(style='whitegrid')
np.random.seed(config.RANDOM_STATE)
plt.rcParams.update({'figure.dpi': config.FIG_DPI})

## 1. Load Features

In [ ]:
df = pd.read_csv(
    os.path.join(config.DATA_DIR, 'sp500_processed.csv'),
    index_col='Date', parse_dates=True
)
# Predict NEXT-day log return
df['LogReturn_next'] = df['LogReturn'].shift(-1)
df = df.dropna()

FEATURES = ['MA_cross', 'Volatility', 'RSI', 'BB_width', 'BB_pct']
X = df[FEATURES]
y = df['LogReturn_next']
print(f'X shape: {X.shape}   y shape: {y.shape}')

## 2. Fit OLS Model

In [ ]:
X_const = sm.add_constant(X)
model = sm.OLS(y, X_const).fit()
print(model.summary())

## 3. In-Sample Predictions

In [ ]:
y_pred = model.fittedvalues
residuals = model.resid

print(f'R²       = {model.rsquared:.4f}')
print(f'Adj. R²  = {model.rsquared_adj:.4f}')
print(f'AIC      = {model.aic:.2f}')
print(f'BIC      = {model.bic:.2f}')

## 4. Residual Diagnostics

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 4a – Residuals vs Fitted
axes[0, 0].scatter(y_pred, residuals, s=8, alpha=0.4, color='steelblue')
axes[0, 0].axhline(0, color='tomato', linewidth=1.5)
axes[0, 0].set_title('Residuals vs Fitted')
axes[0, 0].set_xlabel('Fitted Values')
axes[0, 0].set_ylabel('Residuals')

# 4b – Residual histogram
axes[0, 1].hist(residuals, bins=60, density=True, color='lightsteelblue', edgecolor='white')
x_r = np.linspace(residuals.min(), residuals.max(), 300)
axes[0, 1].plot(x_r, stats.norm.pdf(x_r, residuals.mean(), residuals.std()),
                'tomato', linewidth=2, label='Normal')
axes[0, 1].set_title('Residual Distribution')
axes[0, 1].set_xlabel('Residual')
axes[0, 1].set_ylabel('Density')
axes[0, 1].legend()

# 4c – Q-Q plot
sm.qqplot(residuals, line='s', ax=axes[1, 0], alpha=0.4, markersize=3)
axes[1, 0].set_title('Normal Q-Q Plot of Residuals')

# 4d – Scale-Location
axes[1, 1].scatter(y_pred, np.sqrt(np.abs(residuals)), s=8, alpha=0.4, color='steelblue')
axes[1, 1].set_title('Scale-Location')
axes[1, 1].set_xlabel('Fitted Values')
axes[1, 1].set_ylabel('√|Residuals|')

plt.tight_layout()
plt.show()

## 5. Coefficient Plot

In [ ]:
coefs = model.params.drop('const')
ci    = model.conf_int().drop('const')

fig, ax = plt.subplots(figsize=(8, 5))
y_pos = range(len(coefs))
ax.barh(y_pos, coefs, color=['tomato' if c < 0 else 'steelblue' for c in coefs],
        alpha=0.7, edgecolor='white')
ax.errorbar(coefs, y_pos,
            xerr=[coefs - ci[0], ci[1] - coefs],
            fmt='none', color='black', capsize=4)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_yticks(y_pos)
ax.set_yticklabels(coefs.index)
ax.set_title('OLS Coefficient Estimates with 95% CI')
ax.set_xlabel('Coefficient')
plt.tight_layout()
plt.show()

## 6. Breusch-Pagan & Durbin-Watson Tests

In [ ]:
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.stattools import durbin_watson

bp_stat, bp_p, _, _ = het_breuschpagan(residuals, X_const)
dw = durbin_watson(residuals)
print(f'Breusch-Pagan test  stat={bp_stat:.4f}  p={bp_p:.4e}')
print(f'Durbin-Watson stat  = {dw:.4f}  (2.0 → no autocorrelation)')